# Module 3: Background — LLM Experts, LoRA & Model Merging

**Estimated time: 45 minutes**

This module covers the representation foundation: how LLM experts are stored as LoRA adapters and how the merge operation enables all of Model Swarms' weight-space arithmetic.

## 3.1 The Foundation: What Are LLM Experts?

An "LLM expert" in the context of this paper is a pre-trained language model that has been **fine-tuned** on a specific domain or task. The paper uses 10 initial experts, all based on **Gemma-7B** (Google's 7-billion parameter model), each fine-tuned on a different subset of the Tulu-v2 dataset:

| Expert | Training Domain | What It's Good At |
|--------|----------------|-------------------|
| FLAN | Instruction following | General instruction completion |
| CoT | Chain-of-thought | Step-by-step reasoning |
| LIMA | Curated responses | High-quality, thoughtful answers |
| Open Assistant | Dialogue | Conversational interactions |
| Science | Scientific text | Technical/scientific questions |
| ShareGPT | ChatGPT-like responses | Broad conversational ability |
| Code | Programming | Code generation and explanation |
| Wizardlm | Complex instructions | Multi-step instruction following |
| Ultrachat | Long dialogues | Extended conversational contexts |
| MetaMath | Mathematics | Mathematical problem solving |

Each expert is stored as a **LoRA adapter** — a small set of weight modifications layered on top of the frozen base Gemma-7B model.

## 3.2 LoRA: Low-Rank Adaptation

LoRA (Low-Rank Adaptation, Hu et al. 2021) is a parameter-efficient fine-tuning method. Instead of updating all model weights during fine-tuning, LoRA:

1. **Freezes** the pre-trained model weights $W$
2. **Adds** a low-rank decomposition $\Delta W = BA$ where:
   - $B$ is a matrix of shape $(d, r)$ — the "up projection"
   - $A$ is a matrix of shape $(r, k)$ — the "down projection"
   - $r \ll \min(d, k)$ is the rank (typically 8, 16, or 32)

3. The effective weight becomes $W + \Delta W = W + BA$

### Why LoRA Matters for Model Swarms

For a 7B parameter model, the full weight matrices contain ~7 billion parameters. A LoRA adapter with rank 16 applied to attention layers contains only **~18 million parameters** — a 400x reduction.

This matters because:
- **Storage**: Each "particle" in the swarm is ~70MB instead of ~28GB
- **Merging**: Combining two LoRA adapters is a simple weighted sum of small tensors
- **Speed**: Loading, saving, and arithmetic on 18M parameters is fast
- **Linear interpolation works**: The low-rank structure preserves the property that interpolated weights produce coherent models

### LoRA Arithmetic

If you have two LoRA adapters $\Delta W_1$ and $\Delta W_2$, their weighted combination is:

$$\Delta W_{combined} = \alpha_1 \cdot \Delta W_1 + \alpha_2 \cdot \Delta W_2$$

The resulting model uses $W + \Delta W_{combined}$. This is exactly the operation that Model Swarms performs when updating particle positions.

## 3.3 Understanding the Merge Operation

The `merge.py` file in the Model Swarms codebase implements the core weight combination operation. Let's study and implement it:

In [ ]:
from model_swarms_course.merging import lora_merge_from_paths as lora_merge

print("Imported lora_merge from src/model_swarms_course/merging.py")


This is remarkably simple. The entire merge operation is:

```
for each tensor key in the adapter:
    merged[key] = w1 * adapter1[key] + w2 * adapter2[key] + ...
```

No special handling, no gradient computation, no activation analysis. Just weighted sums of tensors.

## 3.4 How Model Swarms Uses Merging

Every PSO operation in Model Swarms is expressed as a merge:

**Computing velocity components:**
```python
# personal_best - current_position  (cognitive direction)
lora_merge([1, -1], [personal_best_path, current_path], output=p_minus_x)

# global_best - current_position  (social direction)
lora_merge([1, -1], [global_best_path, current_path], output=g_minus_x)

# current_position - global_worst  (repulsion direction)
lora_merge([-1, 1], [global_worst_path, current_path], output=x_minus_gw)
```

**Combining into new velocity:**
```python
lora_merge(
    [inertia_w, cognitive_w, social_w, repel_w],
    [velocity, p_minus_x, g_minus_x, x_minus_gw],
    output=new_velocity
)
```

**Position update:**
```python
lora_merge([1, step_length], [current, velocity], output=new_position)
```

Every single operation — subtraction, addition, scaling — is implemented as a call to `lora_merge` with appropriate weights.

## 3.5 Let's Verify the Merge Operation

Let's create some test adapters and verify that our merge implementation is correct:

In [ ]:
from model_swarms_course.merging import create_dummy_adapter

print("Imported create_dummy_adapter from src/model_swarms_course/merging.py")


In [ ]:
# Test 1: Identity merge — merging a model with itself should return the same model
with tempfile.TemporaryDirectory() as tmpdir:
    adapter_path = os.path.join(tmpdir, "adapter")
    output_path = os.path.join(tmpdir, "output")
    original = create_dummy_adapter(adapter_path)
    
    lora_merge([0.5, 0.5], [adapter_path, adapter_path], output_path)
    
    merged = load_file(os.path.join(output_path, "adapter_model.safetensors"))
    for key in original:
        assert torch.allclose(original[key], merged[key], atol=1e-6), \
            f"Identity merge failed for {key}"

print("PASS: Identity merge (0.5*A + 0.5*A == A)")

In [ ]:
# Test 2: Subtraction — merging [1, -1] with same model should give zeros
with tempfile.TemporaryDirectory() as tmpdir:
    adapter_path = os.path.join(tmpdir, "adapter")
    output_path = os.path.join(tmpdir, "output")
    create_dummy_adapter(adapter_path)
    
    lora_merge([1, -1], [adapter_path, adapter_path], output_path)
    
    merged = load_file(os.path.join(output_path, "adapter_model.safetensors"))
    for key in merged:
        assert torch.allclose(merged[key], torch.zeros_like(merged[key]), atol=1e-6)

print("PASS: Subtraction (A - A == 0)")

In [ ]:
# Test 3: Linearity — merge([0.3], [A]) + merge([0.7], [A]) == merge([1.0], [A])
with tempfile.TemporaryDirectory() as tmpdir:
    adapter_path = os.path.join(tmpdir, "adapter")
    out1 = os.path.join(tmpdir, "out1")
    out2 = os.path.join(tmpdir, "out2")
    out3 = os.path.join(tmpdir, "out3")
    create_dummy_adapter(adapter_path)
    
    lora_merge([0.3], [adapter_path], out1)
    lora_merge([0.7], [adapter_path], out2)
    lora_merge([1.0], [adapter_path], out3)
    
    m1 = load_file(os.path.join(out1, "adapter_model.safetensors"))
    m2 = load_file(os.path.join(out2, "adapter_model.safetensors"))
    m3 = load_file(os.path.join(out3, "adapter_model.safetensors"))
    
    for key in m1:
        assert torch.allclose(m1[key] + m2[key], m3[key], atol=1e-5)

print("PASS: Linearity (0.3*A + 0.7*A == 1.0*A)")

In [ ]:
# Test 4: Velocity computation — velocity = personal_best - current
with tempfile.TemporaryDirectory() as tmpdir:
    pb_path = os.path.join(tmpdir, "personal_best")
    curr_path = os.path.join(tmpdir, "current")
    vel_path = os.path.join(tmpdir, "velocity")
    
    pb = create_dummy_adapter(pb_path, seed=42)
    curr = create_dummy_adapter(curr_path, seed=123)
    
    lora_merge([1, -1], [pb_path, curr_path], vel_path)
    
    vel = load_file(os.path.join(vel_path, "adapter_model.safetensors"))
    for key in pb:
        expected = pb[key] - curr[key]
        assert torch.allclose(expected, vel[key], atol=1e-6)

print("PASS: Velocity computation (personal_best - current)")

In [ ]:
# Test 5: Multi-adapter merge (simulating velocity combination)
with tempfile.TemporaryDirectory() as tmpdir:
    paths = []
    for i in range(4):
        p = os.path.join(tmpdir, f"adapter_{i}")
        create_dummy_adapter(p, seed=i*10)
        paths.append(p)
    
    output = os.path.join(tmpdir, "merged")
    weights = [0.2, 0.3, 0.4, 0.1]  # normalized weights summing to 1
    lora_merge(weights, paths, output)
    
    merged = load_file(os.path.join(output, "adapter_model.safetensors"))
    adapters = [load_file(os.path.join(p, "adapter_model.safetensors")) for p in paths]
    
    for key in merged:
        expected = sum(w * a[key] for w, a in zip(weights, adapters))
        assert torch.allclose(expected, merged[key], atol=1e-5)

print("PASS: 4-adapter weighted merge")

In [ ]:
# Test 6: Position update — new_position = current + step_length * velocity
with tempfile.TemporaryDirectory() as tmpdir:
    curr_path = os.path.join(tmpdir, "current")
    vel_path = os.path.join(tmpdir, "velocity")
    out_path = os.path.join(tmpdir, "new_position")
    
    curr = create_dummy_adapter(curr_path, seed=42)
    vel = create_dummy_adapter(vel_path, seed=99)
    
    step_length = 0.85
    lora_merge([1, step_length], [curr_path, vel_path], out_path)
    
    result = load_file(os.path.join(out_path, "adapter_model.safetensors"))
    for key in curr:
        expected = curr[key] + step_length * vel[key]
        assert torch.allclose(expected, result[key], atol=1e-5)

print("PASS: Position update (current + 0.85 * velocity)")
print("\n=== ALL merge operation tests PASSED ===")

## 3.6 The Model Merging Landscape

Model Swarms exists within a broader ecosystem of model merging techniques:

### Static Methods (Task-Independent)

**Uniform Soup** — Average all model weights equally: $\text{merged} = \frac{1}{N} \sum \text{model}_i$

**SLERP** (Spherical Linear Interpolation) — Interpolate along the surface of a hypersphere, preserving weight vector magnitude.

**DARE-TIES** — Prune small weight changes, resolve sign conflicts by majority vote, then merge survivors.

**Model Stocks** — Use geometric center-of-mass to find a better merge point.

### Dynamic Methods (Task-Dependent)

**Greedy Soup** — Start with the best expert, iteratively try adding each remaining expert with equal weight.

**LoraHub** — Learn mixture coefficients via gradient descent on task data.

**EvolMerge** — Use evolutionary algorithms to search over per-layer merge ratios.

### Where Model Swarms Differs

Model Swarms is **dynamic** (uses task data) but **gradient-free** (unlike LoraHub). It searches **collaboratively** (unlike Greedy Soup) and **continuously** (unlike EvolMerge's discrete mutations).

## 3.7 Population Expansion via Interpolation

The paper starts with 10 initial experts but expands to 20 particles using **pairwise interpolation**:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from model_swarms_course.pso import expand_population

print("Imported expand_population from src/model_swarms_course/pso.py")


## 3.8 Why Linear Weight Interpolation Works

A natural question: why does averaging/interpolating model weights produce good models at all?

### 1. Shared Pre-training Basin
All experts share the same pre-trained base model. Fine-tuning with LoRA makes small perturbations to this base. The experts all live in a relatively small region of weight space.

### 2. Linear Mode Connectivity
Research (Frankle et al., 2020; Neyshabur et al., 2020) has shown that neural networks fine-tuned from the same checkpoint exhibit **linear mode connectivity** — you can linearly interpolate between their weights without encountering a loss barrier.

### 3. Low-Rank Structure
LoRA adapters are low-rank matrices. Interpolating between low-rank matrices produces another low-rank matrix.

### 4. Empirical Support from Model Soups
Wortsman et al. (2022) demonstrated that averaging multiple fine-tuned checkpoints often improves performance over any individual checkpoint.

## Exercise 3.1: Analyze Merge Properties

Explore how the merge operation affects adapter statistics:

In [ ]:
# Explore how merge weights affect the resulting adapter
with tempfile.TemporaryDirectory() as tmpdir:
    adapter_a_path = os.path.join(tmpdir, "adapter_a")
    adapter_b_path = os.path.join(tmpdir, "adapter_b")
    
    adapter_a = create_dummy_adapter(adapter_a_path, seed=42)
    adapter_b = create_dummy_adapter(adapter_b_path, seed=123)
    
    print("Interpolation analysis:")
    print(f"{'Alpha':>6} {'Total Norm':>12} {'Max Abs':>10}")
    print("-" * 32)
    
    for alpha in [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0]:
        output = os.path.join(tmpdir, f"merged_{alpha}")
        lora_merge([alpha, 1 - alpha], [adapter_a_path, adapter_b_path], output)
        merged = load_file(os.path.join(output, "adapter_model.safetensors"))
        
        total_norm = sum(torch.norm(v).item() for v in merged.values())
        max_abs = max(torch.max(torch.abs(v)).item() for v in merged.values())
        print(f"{alpha:6.2f} {total_norm:12.2f} {max_abs:10.4f}")

# Questions to answer:
# 1. How does the total norm change as alpha varies?
# 2. Is the relationship linear?
# 3. What happens when alpha > 1.0 (extrapolation)?

## Exercise 3.2: Conceptual Questions

Answer in the cell below:

1. If all 10 initial experts were trained on the same data split, how would this affect Model Swarms?
2. The population expansion uses weights $t \in [0, 2]$ (allowing extrapolation). Why might this be beneficial?
3. The paper uses LoRA rank 16. What would happen with rank 4? Rank 64?
4. LoRA merging applies the same weight $\alpha$ to every tensor. Could per-layer weights help?

*Your answers here:*

1. 
2. 
3. 
4. 

---

**Next: [Module 4 — The Model Swarms Algorithm: Deep Dive](module_04_algorithm_deep_dive.ipynb)**

---